In [ ]:
import json
from sklearn.model_selection import train_test_split
import pandas as pd
import hashlib
import re
import unicodedata
from pathlib import Path


In [ ]:

df = pd.read_json(
    r"C:\Users\vishn\Downloads\resume_extraction_cleaned.jsonl",
    lines=True
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nTotal records:", len(df))
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

In [ ]:
expected_fields = {
    "current_title",
    "previous_titles",
    "current_company",
    "previous_companies",
    "years_experience",
    "seniority",
    "primary_domain",
    "industries",
    "core_skills",
    "secondary_skills",
    "tools",
    "leadership_experience",
    "key_achievements",
    "location",
    "summary"
}

for index, assistant in enumerate(df["assistant"]):
    assert isinstance(assistant, dict), f"Invalid output at row {index}"
    assert set(assistant.keys()) == expected_fields, f"Schema mismatch at row {index}"

print("All assistant outputs have the expected schema!")

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

print("Tokenizer loaded successfully!")

In [ ]:
def calculate_tokens(row):

    messages = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]},
        {
            "role": "assistant",
            "content": json.dumps(
                row["assistant"],
                ensure_ascii=False
            )
        }
    ]

    encoding = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    # Extract the actual token IDs
    token_ids = encoding["input_ids"]

    # Count tokens
    return len(token_ids)

In [ ]:
df["token_length"] = df.apply(
    calculate_tokens,
    axis=1
)

print(df[["token_length"]].head())
print(df["token_length"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))

In [ ]:
row = df.iloc[0]

messages = [
    {
        "role": "system",
        "content": row["system"]
    },
    {
        "role": "user",
        "content": row["user"]
    },
    {
        "role": "assistant",
        "content": json.dumps(
            row["assistant"],
            ensure_ascii=False
        )
    }
]

token_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

print("Return type:", type(token_ids))
print("Return value type:", type(token_ids).__name__)

if isinstance(token_ids, dict):
    print("Dictionary keys:", token_ids.keys())
    print("Input IDs shape:", token_ids["input_ids"].shape)
else:
    print("Token count:", len(token_ids))
    print("First 20 tokens:", token_ids[:20])

In [ ]:
print(
    df["token_length"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)
longest_resumes = df.nlargest(
    10,
    "token_length"
)

print(
    longest_resumes[
        ["token_length"]
    ]
)

In [ ]:
def calculate_token_breakdown(row):

    # System + user messages
    prompt_messages = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]}
    ]

    # Complete conversation
    full_messages = prompt_messages + [
        {
            "role": "assistant",
            "content": json.dumps(
                row["assistant"],
                ensure_ascii=False
            )
        }
    ]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    return pd.Series({
        "prompt_tokens": len(prompt_ids),
        "total_tokens": len(full_ids),
        "completion_tokens": len(full_ids) - len(prompt_ids)
    })

In [ ]:
limits = [2048, 3072, 4096, 6144, 8192, 10240]

for limit in limits:

    count = (df["token_length"] > limit).sum()

    percentage = count / len(df) * 100

    print(
        f"Limit: {limit:5} | "
        f"Truncated: {count:4} | "
        f"Percentage: {percentage:.2f}%"
    )

In [ ]:
token_breakdown = df.apply(
    calculate_token_breakdown,
    axis=1
)

df = pd.concat([df, token_breakdown], axis=1)

print(df.columns.tolist())

In [ ]:
assert (df["token_length"] == df["total_tokens"]).all()

assert (
    df["prompt_tokens"] + df["completion_tokens"]
    == df["total_tokens"]
).all()

print("All token counts are consistent!")

In [ ]:
print("PROMPT TOKEN STATISTICS")
print(df["prompt_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

print("\nCOMPLETION TOKEN STATISTICS")
print(df["completion_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

In [ ]:

def check_token_alignment(row):
    prompt_messages = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]}
    ]

    full_messages = prompt_messages + [{
        "role": "assistant",
        "content": json.dumps(row["assistant"], ensure_ascii=False)
    }]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    return full_ids[:len(prompt_ids)] == prompt_ids


alignment_results = df.apply(check_token_alignment, axis=1)

print("Aligned:", alignment_results.sum())
print("Not aligned:", (~alignment_results).sum())

In [ ]:
alignment_results = df.apply(check_token_alignment, axis=1)

print("Aligned:", alignment_results.sum())
print("Not aligned:", (~alignment_results).sum())

In [ ]:

def normalize_resume(text):
    if not isinstance(text, str):
        return ""

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Normalize capitalization and whitespace
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()

    return text


def generate_hash(text):
    normalized_text = normalize_resume(text)
    return hashlib.sha256(
        normalized_text.encode("utf-8")
    ).hexdigest()


df["resume_hash"] = df["user"].apply(generate_hash)

In [ ]:
total_records = len(df)

duplicate_mask = df["resume_hash"].duplicated(keep=False)

duplicate_records = duplicate_mask.sum()
unique_resumes = df["resume_hash"].nunique()
duplicate_groups = (
    df.loc[duplicate_mask, "resume_hash"].nunique()
)

print("Total records:", total_records)
print("Unique resumes:", unique_resumes)
print("Records belonging to duplicate groups:", duplicate_records)
print("Number of duplicate groups:", duplicate_groups)

In [ ]:
duplicate_df = df[duplicate_mask].sort_values("resume_hash")

for resume_hash, group in duplicate_df.groupby("resume_hash"):
    print("\nDuplicate group:")
    print("Number of copies:", len(group))
    print("DataFrame indices:", group.index.tolist())
    print("Resume preview:", group.iloc[0]["user"][:300])

In [ ]:
print("Before removing duplicates:", len(df))

df = df.drop_duplicates(
    subset="resume_hash",
    keep="first"
).copy()

df = df.reset_index(drop=True)

print("After removing duplicates:", len(df))
print("Remaining duplicate hashes:", df["resume_hash"].duplicated().sum())

In [ ]:
# First split: 80% training, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Second split: divide temporary data equally
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_df))

In [ ]:
def convert_to_prompt_completion(row):

    prompt_messages = [
        {
            "role": "system",
            "content": row["system"]
        },
        {
            "role": "user",
            "content": row["user"]
        }
    ]

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    completion = json.dumps(
        row["assistant"],
        ensure_ascii=False
    ) + tokenizer.eos_token

    return {
        "prompt": prompt,
        "completion": completion
    }

In [ ]:
train_data = [
    convert_to_prompt_completion(row)
    for _, row in train_df.iterrows()
]

val_data = [
    convert_to_prompt_completion(row)
    for _, row in val_df.iterrows()
]

test_data = [
    convert_to_prompt_completion(row)
    for _, row in test_df.iterrows()
]

In [ ]:
print("PROMPT:")
print(train_data[0]["prompt"])

print("\nCOMPLETION:")
print(train_data[0]["completion"])

In [ ]:
print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Testing examples:", len(test_data))

assert len(train_data) == len(train_df)
assert len(val_data) == len(val_df)
assert len(test_data) == len(test_df)

assert all(
    isinstance(item["prompt"], str)
    and isinstance(item["completion"], str)
    for item in train_data
)

print("Conversion successful!")

In [ ]:

OUTPUT_DIR = Path("data/processed/sft_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

In [ ]:
def save_jsonl(data, file_path):

    with open(file_path, "w", encoding="utf-8") as file:

        for item in data:
            file.write(
                json.dumps(item, ensure_ascii=False) + "\n"
            )

    print(f"Saved {len(data)} records to {file_path}")

In [ ]:
save_jsonl(
    train_data,
    OUTPUT_DIR / "train.jsonl"
)

save_jsonl(
    val_data,
    OUTPUT_DIR / "validation.jsonl"
)

save_jsonl(
    test_data,
    OUTPUT_DIR / "test.jsonl"
)